# Sinus_CFD — nnU-Net training on NasalSeg (Google Colab)

Trains a 3D nnU-Net v2 segmentation model on the NasalSeg dataset (130 head CTs,
labels: background, left/right nasal cavity, nasopharynx, left/right maxillary
sinus). Goal: give the classical HU-threshold pipeline in `sinus_cfd.pipeline`
a learned alternative that can tell nasal-cavity air apart from sinus air
(see `docs/stage1_segmentation_baseline.md` for why that distinction is the
current bottleneck).

**Before running this notebook:**
1. On your machine: `py -3.12 scripts/download_nasalseg.py` then
   `py -3.12 scripts/prepare_nnunet_nasalseg.py --nasalseg-root data`
2. Zip `data/nnUNet_raw/Dataset501_NasalSeg/` and upload the zip to your
   Google Drive (e.g. `MyDrive/sinus_cfd/NasalSeg_nnUNet_raw.zip`).
3. Runtime → Change runtime type → GPU (T4 is fine to start; A100 if you have
   Colab Pro/Pro+ and want it faster).

Colab sessions are ephemeral and can disconnect after ~12h (free tier: often
less, and idles out after ~90 min of inactivity). This notebook checkpoints
training results back to Drive after each run so a disconnect doesn't lose
progress — rerun the training cell with `--c` (continue) if it stops early.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install nnU-Net v2

In [ ]:
!pip install -q nnunetv2>=2.5.0

## 3. Mount Drive and stage the dataset locally

nnU-Net does heavy small-file I/O during preprocessing/training; Drive-mounted
storage is much slower for that than Colab's local disk, so copy the zip in
and extract it to `/content` rather than pointing `nnUNet_raw` at Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this path if you uploaded the zip somewhere else in Drive
DRIVE_ZIP = '/content/drive/MyDrive/sinus_cfd/NasalSeg_nnUNet_raw.zip'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/sinus_cfd/nnUNet_results'

In [ ]:
import os
os.makedirs('/content/nnUNet_data/nnUNet_raw', exist_ok=True)
os.makedirs('/content/nnUNet_data/nnUNet_preprocessed', exist_ok=True)
os.makedirs('/content/nnUNet_data/nnUNet_results', exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

!unzip -q -o "$DRIVE_ZIP" -d /content/nnUNet_data/nnUNet_raw
!ls /content/nnUNet_data/nnUNet_raw

## 4. Set nnU-Net environment variables

In [ ]:
import os
os.environ['nnUNet_raw'] = '/content/nnUNet_data/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_data/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_data/nnUNet_results'
print(os.environ['nnUNet_raw'])
print(os.environ['nnUNet_preprocessed'])
print(os.environ['nnUNet_results'])

## 5. Plan and preprocess (dataset id 501 = NasalSeg)

This resamples all 130 cases to nnU-Net's chosen spacing and computes the
network configuration. Takes a few minutes for a dataset this size.

In [ ]:
!nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity

## 6. Train fold 0 of 3d_fullres

Uses the **250-epoch** trainer variant, not nnU-Net's 1000-epoch default —
`nnUNetTrainer_250epochs` rescales the whole polynomial LR schedule to 250
epochs rather than stopping early, so it's a proper (shorter) training run,
not a truncated one. On a T4 at ~22 s/epoch that's ~1.5 h instead of ~50 h,
and 250 epochs is plenty for a 130-case MVP baseline (104 training cases per
fold — the 1000-epoch schedule mostly matters for larger datasets).

If the runtime disconnects partway through, rerun this same cell with `--c`
appended to resume from the last checkpoint instead of restarting.

MVP scope: **fold 0 only** (not the full 5-fold CV) — enough for a usable
model and a held-out validation Dice. Add folds 1-4 later for a proper
cross-validated estimate. Note the trainer name becomes part of the results
path, so this run won't collide with any earlier default-trainer run.

In [ ]:
# 250-epoch trainer: ~4x faster than the 1000-epoch default (rescales the LR
# schedule, doesn't just truncate it). Plenty for a 130-case MVP baseline.
# If the runtime disconnects mid-run, rerun this SAME cell with --c to resume.
!nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainer_250epochs

## 7. Copy trained weights back to Drive

Colab's local disk does not persist across sessions — do this before the
runtime is recycled.

In [ ]:
!rsync -av /content/nnUNet_data/nnUNet_results/ "$DRIVE_RESULTS_DIR/"
print('Synced to', DRIVE_RESULTS_DIR)

## 8. (Optional) Validation Dice summary

nnU-Net writes per-case validation metrics under
`nnUNet_results/Dataset501_NasalSeg/.../fold_0/validation/summary.json`.

In [ ]:
import glob, json
summaries = glob.glob('/content/nnUNet_data/nnUNet_results/Dataset501_NasalSeg/**/summary.json', recursive=True)
print(summaries)
if summaries:
    s = json.load(open(summaries[0]))
    print(json.dumps(s.get('foreground_mean', s), indent=2))

## Next step back in the repo

During training's final validation, nnU-Net writes predicted label maps for
its **held-out fold-0 cases** to
`nnUNet_results/Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/validation/`.
Cell 7's rsync already copies these to Drive — they're exactly the fair
(never-trained-on) predictions to score against the classical baseline. No
need to re-run `nnUNetv2_predict` for the comparison.

Back on your machine:

1. Download that `fold_0/validation/` folder from Drive (e.g. to
   `data/nnUNet_results/.../fold_0/validation/`).
2. Run the head-to-head comparison — same ground truth (labels 1-3) and same
   Dice as `docs/stage1_segmentation_baseline.md`, so the number is directly
   comparable to the ~0.25 classical baseline:

   ```powershell
   py -3.12 scripts/compare_nnunet_vs_classical.py `
       --pred-dir data/nnUNet_results/.../fold_0/validation `
       --nasalseg-root data
   ```

   It prints per-case nnU-Net vs classical airway Dice, the mean improvement,
   and nnU-Net's per-structure Dice (so you can see whether it cleanly
   separates nasal cavity from maxillary sinus — the thing the classical
   threshold pipeline couldn't do).

To predict on a brand-new CT later (not a NasalSeg case), use the trained
weights directly:

```
nnUNetv2_predict -i INPUT_FOLDER -o OUTPUT_FOLDER -d 501 -c 3d_fullres -f 0 -tr nnUNetTrainer_250epochs
```